# Zero-to-Hero: Production-Grade AI SQL Assistant with LangChain + Ollama

This notebook builds and explains a full local AI SQL analytics assistant.

## Learning Objectives
- Understand architecture for enterprise-style NL-to-SQL assistants.
- Build schema-aware SQL generation with LangChain and direct prompting.
- Add read-only safety validation and guardrails.
- Execute SQL, explain it, and visualize results.
- Benchmark and evaluate model quality with LLM-as-a-judge.


## Why These Choices?

### Why SQLite for learning?
- Fully local, reproducible, zero cloud dependency.
- Simple setup with strong SQL feature coverage for analytics.
- Easy packaging for tutorials and CI.

### Why LangChain `SQLDatabase`?
- Standard schema-aware abstraction.
- Faster bootstrapping for text-to-SQL pipelines.
- Good baseline for comparison against direct prompting.

### Why Ollama?
- Local inference for privacy and cost control.
- Deterministic settings possible (`temperature=0`, fixed seed).
- Easy model swapping (`qwen3.5:4b`, `granite4.1:3b`).

### Why deterministic decoding (`temperature=0`)?
- SQL must be reproducible for audits and regression tests.
- Lower syntax drift and fewer random unsafe tokens.
- Stable benchmark comparisons and easier debugging.

### Why read-only access?
- Protects production-like data from destructive writes.
- Enforces analytics-only assistant behavior.
- Required for enterprise governance.

### Why schema-aware prompting?
- Reduces hallucinated tables/columns.
- Improves join accuracy and KPI grounding.
- Enables business-term to schema mapping.


## System Architecture

```mermaid
flowchart LR
    User[User Question] --> UI[Streamlit App]
    UI --> Pipeline[AISQLAssistant]
    Pipeline --> Schema[Schema Cache + Glossary]
    Pipeline --> GenLC[LangChain SQL Generation]
    Pipeline --> GenDirect[Direct Ollama SQL Generation]
    GenLC --> Validate[SQL Validation Guardrails]
    GenDirect --> Validate
    Validate --> Execute[Read-only SQLite Execution]
    Execute --> Explain[SQL Explanation + Business Interpretation]
    Execute --> Viz[Visualization Recommender]
    Pipeline --> Eval[Benchmark + Judge]
    Pipeline --> Memory[Conversation + History Store]
```


In [ ]:
# 1) Imports and runtime setup
from ai_sql_assistant.logging_utils import configure_logging
from ai_sql_assistant.config import get_settings
from ai_sql_assistant.data.northwind import build_northwind_databases
from ai_sql_assistant.schema.introspector import inspect_database, save_schema_report, generate_erd
from ai_sql_assistant.pipeline.assistant import AISQLAssistant
from ai_sql_assistant.types import QueryRequest

configure_logging()
settings = get_settings()
settings


In [ ]:
# 2) Build realistic relational dataset (Northwind raw + scaled)
# This step attempts public Northwind download first; falls back to deterministic synthetic build.
build_result = build_northwind_databases(
    raw_db_path=settings.database.raw_db_path,
    scaled_db_path=settings.database.scaled_db_path,
    scale_factor=8,
)
build_result


In [ ]:
# 3) Automatic database exploration and schema report generation
report = inspect_database(settings.database.active_db_path)
md_path, json_path = save_schema_report(report)
erd_path = generate_erd(report)

print(f"Schema markdown: {md_path}")
print(f"Schema json: {json_path}")
print(f"ERD image: {erd_path}")
print("Tables:", list(report['tables'].keys()))


### What we explored automatically
- Tables, columns, datatypes.
- Primary keys and foreign keys.
- Relationship graph and indexes.
- Row counts and null statistics.
- Sample rows for quick inspection.


In [ ]:
# 4) Initialize assistant pipeline
assistant = AISQLAssistant(settings)


In [ ]:
# 5) Ask a business question end-to-end
request = QueryRequest(
    question="Show monthly net revenue for Europe in 2024.",
    persona="Finance",
    user_id="notebook",
    conversation_id="nb-session",
)
response = assistant.ask(request, approach="langchain", model=settings.models.generator_model)

print("Status:", response.execution.status)
print("SQL:\n", response.execution.sql)
print("Rows:", response.execution.row_count)
print("Execution ms:", response.execution.execution_time_ms)


In [ ]:
# 6) Validation details
for issue in response.validation.issues:
    print(issue.code, "=>", issue.message)


In [ ]:
# 7) Explanation output for beginners
print(response.explanation[:2000])


In [ ]:
# 8) Visualize query result candidates
import pandas as pd
from ai_sql_assistant.visualization.recommender import render_chart

result_df = pd.DataFrame(response.execution.rows)
result_df.head()


In [ ]:
# Try first non-table chart recommendation
if response.visualization_options:
    spec = next((s for s in response.visualization_options if s.chart_type != "table"), response.visualization_options[0])
    fig = render_chart(response.execution.rows, spec)
    fig


## Prompt Engineering Personas
This project includes persona templates:
- Business Analyst
- Finance
- Sales
- HR
- Inventory
- Marketing

Each template shifts SQL priorities while preserving safety rules.


In [ ]:
# 9) Compare LangChain SQL generation vs direct Ollama prompting
question = "Top 10 customers by revenue in Germany."
req = QueryRequest(question=question, persona="Sales", user_id="notebook", conversation_id="nb-compare")

res_langchain = assistant.ask(req, approach="langchain", model=settings.models.generator_model)
res_direct = assistant.ask(req, approach="direct", model=settings.models.generator_model)

print("LangChain SQL:\n", res_langchain.execution.sql)
print("\nDirect SQL:\n", res_direct.execution.sql)


## Benchmark and Evaluation
Benchmark set contains 100 business questions with ground-truth SQL.

Metrics tracked:
- Exact SQL match (normalized)
- Execution accuracy
- Result correctness
- Generation latency
- Execution latency
- Complexity score
- LLM judge rubric (correctness, safety, readability, efficiency)


In [ ]:
# 10) Run benchmark (heavy). Uncomment to execute full matrix.
# from pathlib import Path
# from ai_sql_assistant.benchmarking.benchmark import BenchmarkRunner
# runner = BenchmarkRunner(assistant, settings)
# cases = runner.load_cases(Path("benchmarks/benchmark_cases.json"))
# run = runner.run(cases)
# judge = runner.evaluate_with_judge({c.case_id: c for c in cases}, run)
# out_path = runner.save_run(run, judge)
# print(out_path)
# runner.close()


## Failure Analysis
Failure suite covers:
- Hallucinated tables
- Hallucinated columns
- Missing joins
- Ambiguous requests
- Unsafe SQL / prompt injection attempts

Use script:
```bash
uv run python scripts/run_failure_analysis.py
```


## Next Steps
- Launch Streamlit app:
```bash
uv run streamlit run streamlit_app/app.py
```
- Run tests:
```bash
uv run pytest -v
```
- Run end-to-end pipeline:
```bash
uv run python scripts/run_end_to_end.py
```


In [ ]:
# Cleanup resources at end of notebook
assistant.close()
